Environment Check.

In [ ]:
# Quick SageMaker preflight — checks packages, AWS identity, S3 access, base path, HF streaming
import os
import importlib

print("=== OpenFake SageMaker Preflight Check ===\n")

# 1) Package checks
required = ["boto3", "tqdm", "datasets"]
missing = []

for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"[OK] Package available: {pkg}")
    except Exception as e:
        print(f"[MISSING] {pkg} -> {e}")
        missing.append(pkg)

# 2) Base path check
base_dir = os.environ.get("OPENFAKE_BASE_DIR", "/home/ec2-user/SageMaker")
print(f"\n[INFO] OPENFAKE_BASE_DIR = {base_dir}")
print(f"[INFO] Base path exists: {os.path.exists(base_dir)}")

# 3) AWS identity + S3 bucket check
aws_ok = True
bucket_name = "deepfake-d-100k-dataset-tw26"

try:
    import boto3

    sts = boto3.client("sts")
    ident = sts.get_caller_identity()
    print(f"\n[OK] AWS identity detected")
    print(f"     Account: {ident.get('Account')}")
    print(f"     ARN: {ident.get('Arn')}")

    s3 = boto3.client("s3")
    s3.head_bucket(Bucket=bucket_name)
    print(f"[OK] S3 bucket accessible: {bucket_name}")

except Exception as e:
    aws_ok = False
    print(f"\n[FAIL] AWS/S3 check failed: {e}")

# 4) Hugging Face streaming probe
hf_ok = True
try:
    from datasets import load_dataset

    ds = load_dataset("ComplexDataLab/OpenFake", split="train", streaming=True)
    first = next(iter(ds))
    print(f"\n[OK] Hugging Face streaming works")
    print(f"     Sample keys: {list(first.keys())}")

except Exception as e:
    hf_ok = False
    print(f"\n[FAIL] Hugging Face streaming failed: {e}")

# 5) Summary
print("\n=== Summary ===")
if missing:
    print(f"[ACTION] Install missing packages: {missing}")
else:
    print("[OK] No missing core packages")

if not os.path.exists(base_dir):
    print("[ACTION] Base directory does not exist. Set OPENFAKE_BASE_DIR to a valid writable path.")

if not aws_ok:
    print("[ACTION] Fix AWS credentials / IAM role / S3 bucket permissions before running downloader.")

if not hf_ok:
    print("[ACTION] Fix Hugging Face connectivity or package issues before running downloader.")

if (not missing) and os.path.exists(base_dir) and aws_ok and hf_ok:
    print("\nGREEN: Safe to run the downloader.")
else:
    print("\nNOT READY: Resolve the issues above first.")

In [1]:
%pip install datasets

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 29.1 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 38.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 109.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [datasets]5/6 [datasets]ce-hub]
Note: you may need to restart the kernel to use updated packages.


Download Script for OpenFake - 100K RAW.

HF TOKEN Setup.

In [8]:
# OpenFake AWS Production Downloader.

# Architecture:
#   - Single Hugging Face streaming session downloads both classes in one pass
#   - Real and fake saved separately to local SSD
#   - After download: real folder zipped → uploaded → verified → deleted
#                     fake folder zipped → uploaded → verified → deleted
#   - Separate manifests per class uploaded to S3
#   - base_dir deleted only after both uploads are fully verified

import os
os.environ['HF_DATASETS_DISABLE_PROGRESS_BARS'] = '1'

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is not set in this kernel session.")

from io import BytesIO
from PIL import Image as PILImage, UnidentifiedImageError
import shutil
import random
import json
import boto3
from datetime import datetime, timezone
from datasets import load_dataset
from tqdm.auto import tqdm

random.seed(42)

# CONFIGURATION  —  change these as needed, everything else is derived from them.

TARGET_PER_CLASS   = 50000          # 50K real + 50K fake = 100K raw total
MAX_ITERATIONS     = 600000         # safety ceiling for the HF stream loop
CHECKPOINT_EVERY   = 1000           # write progress checkpoint every N images saved
DISK_MARGIN_FACTOR = 2.2            # require 2.2× the folder size free before zipping

BASE_DIR  = os.environ.get('OPENFAKE_BASE_DIR', '/home/ec2-user/SageMaker')

S3_BUCKET = 'deepfake-d-100k-dataset-tw26'
S3_PREFIX = 'datasets/OpenFake'

# DERIVED PATHS  —  do not hardcode elsewhere

TEMP_RAW_DIR   = os.path.join(BASE_DIR, 'temp_raw')
REAL_DIR       = os.path.join(TEMP_RAW_DIR, 'real')
FAKE_DIR       = os.path.join(TEMP_RAW_DIR, 'fake')
CHECKPOINT_FILE = os.path.join(BASE_DIR, 'openfake_checkpoint.json')

S3_KEYS = {
    'real_zip'       : f'{S3_PREFIX}/openfake_real_raw.zip',
    'fake_zip'       : f'{S3_PREFIX}/openfake_fake_raw.zip',
    'real_manifest'  : f'{S3_PREFIX}/openfake_real_manifest.txt',
    'fake_manifest'  : f'{S3_PREFIX}/openfake_fake_manifest.txt',
}

LOCAL_ZIPS = {
    'real' : os.path.join(BASE_DIR, 'openfake_real_raw'),   # .zip appended by make_archive
    'fake' : os.path.join(BASE_DIR, 'openfake_fake_raw'),
}

MANIFEST_PATHS = {
    'real' : os.path.join(BASE_DIR, 'openfake_real_manifest.txt'),
    'fake' : os.path.join(BASE_DIR, 'openfake_fake_manifest.txt'),
}

# S3 CLIENT

s3 = boto3.client('s3')

# HELPER — CHECKPOINT

def write_checkpoint(iteration, real_count, fake_count, save_errors):
    data = {
        'timestamp'   : datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'),
        'iteration'   : iteration,
        'real_count'  : real_count,
        'fake_count'  : fake_count,
        'save_errors' : save_errors,
    }
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(data, f, indent=2)


def clear_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

# HELPER — DISK SPACE CHECK

def get_folder_size(folder):
    total = 0
    for dirpath, _, filenames in os.walk(folder):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            try:
                total += os.path.getsize(fp)
            except OSError:
                pass
    return total


def check_disk_space_for_zip(folder):
    folder_size   = get_folder_size(folder)
    required      = int(folder_size * DISK_MARGIN_FACTOR)
    _, _, free    = shutil.disk_usage(BASE_DIR)

    folder_gb   = folder_size / (1024 ** 3)
    required_gb = required    / (1024 ** 3)
    free_gb     = free        / (1024 ** 3)

    print(f"  Folder size  : {folder_gb:.2f} GB")
    print(f"  Required free: {required_gb:.2f} GB  (folder × {DISK_MARGIN_FACTOR})")
    print(f"  Actual free  : {free_gb:.2f} GB")

    if free < required:
        raise RuntimeError(
            f"Insufficient disk space before zipping.\n"
            f"  Folder  : {folder_gb:.2f} GB\n"
            f"  Required: {required_gb:.2f} GB\n"
            f"  Free    : {free_gb:.2f} GB\n"
            f"Aborting to prevent archive corruption."
        )

# HELPER — ZIP, UPLOAD, VERIFY, CLEANUP

def create_zip(folder, zip_base_path):
    """Zip a single folder. Returns the .zip path."""
    print(f"  Archiving {os.path.basename(folder)}/  →  {os.path.basename(zip_base_path)}.zip")
    shutil.make_archive(zip_base_path, 'zip', os.path.dirname(folder), os.path.basename(folder))
    zip_path = zip_base_path + '.zip'

    if not os.path.exists(zip_path) or os.path.getsize(zip_path) == 0:
        raise RuntimeError(f"Zip creation failed or produced empty archive: {zip_path}")

    size_gb = os.path.getsize(zip_path) / (1024 ** 3)
    print(f"  Archive ready: {size_gb:.2f} GB")
    return zip_path


def upload_to_s3(local_path, s3_key):
    """Upload a file to S3."""
    size_gb = os.path.getsize(local_path) / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)  →  s3://{S3_BUCKET}/{s3_key}")
    s3.upload_file(local_path, S3_BUCKET, s3_key)


def verify_s3_upload(local_path, s3_key):
    """Confirm S3 object exists and its size matches local file."""
    local_size = os.path.getsize(local_path)
    try:
        response  = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)
        s3_size   = response['ContentLength']
    except Exception as e:
        raise RuntimeError(f"S3 verification failed — head_object error: {e}")

    if s3_size != local_size:
        raise RuntimeError(
            f"S3 size mismatch for {s3_key}.\n"
            f"  Local : {local_size} bytes\n"
            f"  S3    : {s3_size} bytes\n"
            f"Upload may be incomplete. Local files preserved."
        )
    print(f"  Verified: S3 object size matches local ({s3_size / (1024**3):.2f} GB)")


def build_manifest(label, real_count, fake_count, save_errors,
                   error_samples, iteration_counter, zip_path, s3_key):
    count     = real_count if label == 'real' else fake_count
    size_gb   = os.path.getsize(zip_path) / (1024 ** 3) if os.path.exists(zip_path) else 0.0
    timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

    error_block = ''
    if error_samples:
        error_block = '\nSampled error messages (first 10):\n'
        for i, msg in enumerate(error_samples, 1):
            error_block += f'  [{i:02d}] {msg}\n'

    return (
        f"OpenFake Production Download — {label.upper()} Class Manifest\n"
        f"{'=' * 60}\n"
        f"  Timestamp         : {timestamp}\n"
        f"  Class             : {label}\n"
        f"  Target per class  : {TARGET_PER_CLASS}\n"
        f"  Actual count      : {count}\n"
        f"  Save errors       : {save_errors}\n"
        f"  Iterations used   : {iteration_counter}\n"
        f"  Archive size      : {size_gb:.2f} GB\n"
        f"  S3 destination    : s3://{S3_BUCKET}/{s3_key}\n"
        f"{'=' * 60}\n"
        f"{error_block}"
    )


def process_class(label, folder, zip_base, manifest_path,
                  zip_s3_key, manifest_s3_key,
                  real_count, fake_count, save_errors,
                  error_samples, iteration_counter):
    """
    Full post-download pipeline for one class:
    disk check → zip → upload zip → verify → upload manifest → verify → cleanup
    """
    print(f"\n{'─' * 60}")
    print(f"Processing class: {label.upper()}")
    print(f"{'─' * 60}")

    # 1. Disk space check
    print("\n[1/5] Checking disk space...")
    check_disk_space_for_zip(folder)

    # 2. Zip
    print("\n[2/5] Creating archive...")
    zip_path = create_zip(folder, zip_base)

    # 3. Upload zip + verify
    print("\n[3/5] Uploading archive to S3...")
    try:
        upload_to_s3(zip_path, zip_s3_key)
        verify_s3_upload(zip_path, zip_s3_key)
    except Exception as e:
        print(f"\nUpload/verification failed — local files preserved.\n  {e}")
        raise

    # 4. Write + upload manifest
    print("\n[4/5] Writing and uploading manifest...")
    manifest_text = build_manifest(
        label, real_count, fake_count, save_errors,
        error_samples, iteration_counter, zip_path, zip_s3_key
    )
    with open(manifest_path, 'w') as f:
        f.write(manifest_text)
    print(manifest_text)

    try:
        upload_to_s3(manifest_path, manifest_s3_key)
        verify_s3_upload(manifest_path, manifest_s3_key)
    except Exception as e:
        print(f"\nManifest upload failed — local files preserved.\n  {e}")
        raise

    # 5. Cleanup this class only — only reached if both uploads verified
    print("\n[5/5] Cleaning up local artifacts for this class...")
    shutil.rmtree(folder, ignore_errors=True)
    os.remove(zip_path)
    os.remove(manifest_path)
    print(f"  Removed: {folder}")
    print(f"  Removed: {zip_path}")
    print(f"  Removed: {manifest_path}")

# SETUP

print("\n" + "═" * 60)
print("  OpenFake AWS Production Downloader")
print(f"  Target : {TARGET_PER_CLASS:,} real  +  {TARGET_PER_CLASS:,} fake  =  {TARGET_PER_CLASS * 2:,} total raw images")
print(f"  Bucket : s3://{S3_BUCKET}/{S3_PREFIX}/")
print("═" * 60 + "\n")

shutil.rmtree(TEMP_RAW_DIR, ignore_errors=True)
os.makedirs(REAL_DIR, exist_ok=True)
os.makedirs(FAKE_DIR, exist_ok=True)

# DATASET CONNECTION

print("[Setup] Connecting to ComplexDataLab/OpenFake stream...")
try:
    openfake = load_dataset(
        "ComplexDataLab/OpenFake",
        split='train',
        streaming=True,
        token=hf_token
    )
    openfake = openfake.decode(False)   # disable automatic image decoding
    print("[Setup] Connection successful.\n")
except Exception as e:
    raise RuntimeError(f"Dataset loading failed: {e}")

# DOWNLOAD LOOP

iteration_counter = 0
save_errors       = 0
error_samples     = []    # first 10 exception messages for manifest postmortem
real_count        = 0
fake_count        = 0
last_checkpoint   = 0     # tracks images saved since last checkpoint write

pbar_real = tqdm(total=TARGET_PER_CLASS, desc="  Real", unit="img", mininterval=1.0)
pbar_fake = tqdm(total=TARGET_PER_CLASS, desc="  Fake", unit="img", mininterval=1.0)

print("[Download] Starting single-pass stream...\n")

for item in openfake:
    iteration_counter += 1

    if iteration_counter > MAX_ITERATIONS:
        print("\n[Download] Iteration ceiling reached — stream stopped.")
        break

    image = None

    try:
        raw_label = item['label']
        raw_image = item.get('image', None)

        if raw_image is None:
            continue

        # -battle tested from sandbox-
        if isinstance(raw_label, int):
            label = 'real' if raw_label == 0 else 'fake'
        else:
            raw_label_str = str(raw_label).lower().strip()
            if 'real' in raw_label_str:
                label = 'real'
            elif any(x in raw_label_str for x in ['fake', 'sd', 'flux', 'mj', 'midjourney']):
                label = 'fake'
            else:
                continue

        # manual image decode so corrupt bytes can be skipped safely
        if isinstance(raw_image, dict):
            img_bytes = raw_image.get("bytes")
            img_path  = raw_image.get("path")

            if img_bytes is not None:
                image = PILImage.open(BytesIO(img_bytes))
            elif img_path:
                image = PILImage.open(img_path)
            else:
                continue
        else:
            image = raw_image

        image.load()

        if image.mode != 'RGB':
            image = image.convert('RGB')

        if label == 'real' and real_count < TARGET_PER_CLASS:
            filename = f"raw_openfake_real_{real_count:05d}.jpg"
            image.save(os.path.join(REAL_DIR, filename), format='JPEG', quality=95)
            real_count += 1
            pbar_real.update(1)

        elif label == 'fake' and fake_count < TARGET_PER_CLASS:
            filename = f"raw_openfake_fake_{fake_count:05d}.jpg"
            image.save(os.path.join(FAKE_DIR, filename), format='JPEG', quality=95)
            fake_count += 1
            pbar_fake.update(1)

    except (UnidentifiedImageError, OSError, ValueError) as e:
        save_errors += 1
        if len(error_samples) < 10:
            error_samples.append(f"iter={iteration_counter} | {type(e).__name__}: {str(e)[:120]}")
        continue

    except Exception as e:
        save_errors += 1
        if len(error_samples) < 10:
            error_samples.append(f"iter={iteration_counter} | {type(e).__name__}: {str(e)[:120]}")
        continue

    finally:
        try:
            if image is not None:
                image.close()
        except Exception:
            pass

    # -Periodic checkpoint-
    total_saved = real_count + fake_count
    if total_saved - last_checkpoint >= CHECKPOINT_EVERY:
        write_checkpoint(iteration_counter, real_count, fake_count, save_errors)
        last_checkpoint = total_saved

    if real_count == TARGET_PER_CLASS and fake_count == TARGET_PER_CLASS:
        break

pbar_real.close()
pbar_fake.close()

print(f"\n[Download] Complete.")
print(f"  Real      : {real_count:,} / {TARGET_PER_CLASS:,}")
print(f"  Fake      : {fake_count:,} / {TARGET_PER_CLASS:,}")
print(f"  Iterations: {iteration_counter:,}")
print(f"  Errors    : {save_errors}")

# FAIL HARD — do not proceed if targets not met

if real_count < TARGET_PER_CLASS or fake_count < TARGET_PER_CLASS:
    raise RuntimeError(
        f"\nTarget not reached — aborting. No zipping or uploading will occur.\n"
        f"  Real : {real_count:,} / {TARGET_PER_CLASS:,}\n"
        f"  Fake : {fake_count:,} / {TARGET_PER_CLASS:,}"
    )

clear_checkpoint()

# POST-DOWNLOAD PIPELINE — real class first, then fake

for label in ['real', 'fake']:
    process_class(
        label          = label,
        folder         = REAL_DIR         if label == 'real' else FAKE_DIR,
        zip_base       = LOCAL_ZIPS[label],
        manifest_path  = MANIFEST_PATHS[label],
        zip_s3_key     = S3_KEYS[f'{label}_zip'],
        manifest_s3_key= S3_KEYS[f'{label}_manifest'],
        real_count     = real_count,
        fake_count     = fake_count,
        save_errors    = save_errors,
        error_samples  = error_samples,
        iteration_counter = iteration_counter,
    )
clear_checkpoint()

# FINAL CLEANUP — base temp_raw only after both classes fully verified

shutil.rmtree(TEMP_RAW_DIR, ignore_errors=True)
print(f"\n[Cleanup] Removed base staging dir: {TEMP_RAW_DIR}")

print("\n" + "═" * 60)
print("  OpenFake production download — ALL DONE")
print(f"  Real zip : s3://{S3_BUCKET}/{S3_KEYS['real_zip']}")
print(f"  Fake zip : s3://{S3_BUCKET}/{S3_KEYS['fake_zip']}")
print("═" * 60 + "\n")


════════════════════════════════════════════════════════════
  OpenFake AWS Production Downloader
  Target : 50,000 real  +  50,000 fake  =  100,000 total raw images
  Bucket : s3://deepfake-d-100k-dataset-tw26/datasets/OpenFake/
════════════════════════════════════════════════════════════

[Setup] Connecting to ComplexDataLab/OpenFake stream...
[Setup] Connection successful.



  Real:   0%|          | 0/50000 [00:00<?, ?img/s]

  Fake:   0%|          | 0/50000 [00:00<?, ?img/s]

[Download] Starting single-pass stream...


[Download] Complete.
  Real      : 50,000 / 50,000
  Fake      : 50,000 / 50,000
  Iterations: 100,056
  Errors    : 1

────────────────────────────────────────────────────────────
Processing class: REAL
────────────────────────────────────────────────────────────

[1/5] Checking disk space...
  Folder size  : 6.18 GB
  Required free: 13.59 GB  (folder × 2.2)
  Actual free  : 447.37 GB

[2/5] Creating archive...
  Archiving real/  →  openfake_real_raw.zip
  Archive ready: 5.88 GB

[3/5] Uploading archive to S3...
  Uploading openfake_real_raw.zip  (5.88 GB)  →  s3://deepfake-d-100k-dataset-tw26/datasets/OpenFake/openfake_real_raw.zip
  Verified: S3 object size matches local (5.88 GB)

[4/5] Writing and uploading manifest...
OpenFake Production Download — REAL Class Manifest
  Timestamp         : 2026-03-23 14:07:19 UTC
  Class             : real
  Target per class  : 50000
  Actual count      : 50000
  Save errors       : 1
  Iterations used 

Cleanup Cell if the Download gets Interrupted.

In [7]:
import os, shutil

base_dir = os.environ.get("OPENFAKE_BASE_DIR", "/home/ec2-user/SageMaker")
temp_raw_dir = os.path.join(base_dir, "temp_raw")

if os.path.exists(temp_raw_dir):
    shutil.rmtree(temp_raw_dir, ignore_errors=True)
    print(f"Removed: {temp_raw_dir}")
else:
    print("No temp_raw directory found.")

# optional: also remove any leftover local zips/manifests/checkpoints if your script uses them
for name in [
    "openfake_real_raw.zip",
    "openfake_fake_raw.zip",
    "openfake_checkpoint.json",
    "openfake_real_manifest.txt",
    "openfake_fake_manifest.txt",
]:
    path = os.path.join(base_dir, name)
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed: {path}")

Removed: /home/ec2-user/SageMaker/temp_raw
